# Access GCS Bucket

In [1]:
# Run this cell first

import os
from google.colab import userdata
import json

# --- 1. Securely get the key from Colab Secrets ---
key_string = userdata.get('GCP_KEY')
key_data = json.loads(key_string)

# --- 2. Authenticate gsutil ---
# Write the key to a temporary file that gsutil can read
with open('temp_key.json', 'w') as f:
    json.dump(key_data, f)

#os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'temp_key.json'
!gcloud auth activate-service-account --key-file=temp_key.json

# --- 3. Access bucket directories ---
BUCKET_NAME = "asvspoof-la-data-bucket"
!gsutil ls gs://{BUCKET_NAME}

Activated service account credentials for: [colab-storage-reader@voiceauth-479800.iam.gserviceaccount.com]
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_dev/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_eval/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_train/


# Datasets

### Training

In [ ]:
# -- Training Dataset Labels --

import pandas as pd
import os

# --- 1. Define Your Bucket and File Paths ---
BUCKET_NAME = "asvspoof-la-data-bucket" # Or your actual bucket name
PROTOCOL_DIR = "ASVspoof2019_LA_cm_protocols"
TRAIN_PROTOCOL_FILE = "ASVspoof2019.LA.cm.train.trn.txt"
KEY_FILE_NAME = 'temp_key.json'

# Construct the full GCS path
# gsutil paths are gs://BUCKET_NAME/OBJECT_NAME
gcs_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/{TRAIN_PROTOCOL_FILE}"
print(f"Reading labels from: {gcs_path}\n")

storage_options = {'token': KEY_FILE_NAME}

# --- 2. Read the Protocol File with Pandas ---
# [cite_start]Based on the README, the columns are space-separated[cite: 5].
# We'll give them names based on the README's 5-column format:
# [cite_start]SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY [cite: 5]
column_names = ['SPEAKER_ID', 'AUDIO_FILE_NAME', 'BLANK', 'SYSTEM_ID', 'KEY']

# Read the file directly from GCS
# We use sep=' ' to indicate space-separated values
train_df = pd.read_csv(
    gcs_path,
    sep=' ',
    header=None,
    names=column_names,
    storage_options=storage_options,
)

# --- 3. Clean Up and Process the DataFrame ---

# We only need the file name and the label (KEY)
train_df = train_df[['AUDIO_FILE_NAME', 'KEY']]

# Convert string labels ('bonafide', 'spoof') to numbers (0, 1)
# This is a critical step for training.
train_df['LABEL'] = train_df['KEY'].apply(
    lambda x: 0 if x == 'bonafide' else 1
)

# We can now drop the original 'KEY' column
train_df = train_df.drop(columns=['KEY'])

# --- 4. Inspect the Result ---
print("Successfully loaded and processed labels:")
print(train_df.info())
print("\nFirst 5 rows:")
print(train_df.head())
print(f"\nTotal bonafide (real) samples: {(train_df['LABEL'] == 0).sum()}")
print(f"Total spoof (fake) samples: {(train_df['LABEL'] == 1).sum()}")

!rm temp_key.json

Reading labels from: gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt

Successfully loaded and processed labels:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25380 entries, 0 to 25379
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   AUDIO_FILE_NAME  25380 non-null  object
 1   LABEL            25380 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 396.7+ KB
None

First 5 rows:
  AUDIO_FILE_NAME  LABEL
0    LA_T_1138215      0
1    LA_T_1271820      0
2    LA_T_1272637      0
3    LA_T_1276960      0
4    LA_T_1341447      0

Total bonafide (real) samples: 2580
Total spoof (fake) samples: 22800


In [ ]:
# Uninstall any conflicting versions
!pip uninstall -y tensorflow tensorflow-io

# Install the compatible pair for your new Colab runtime
!pip install tensorflow==2.16.1
!pip install tensorflow-io==0.37.1
!pip install ml_dtypes --upgrade

In [ ]:
# -- Training Dataset Spectrograms --

import tensorflow as tf
import tensorflow_io as tfio  # <--- CHANGE 1: ADD THIS IMPORT
import numpy as np

# --- 1. Define Constants ---
BUCKET_NAME = "asvspoof-la-data-bucket"
AUDIO_DIR = "ASVspoof2019_LA_train"
SAMPLE_RATE = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

# --- 2. Create the list of file paths and labels ---
audio_files = train_df['AUDIO_FILE_NAME'].tolist()
labels = train_df['LABEL'].tolist()

# --- 3. Build the tf.data.Dataset ---
dataset = tf.data.Dataset.from_tensor_slices((audio_files, labels))

# --- 4. Define the Preprocessing Function ---
def load_and_preprocess(audio_file_name, label):
    # 4a. Construct the full GCS path
    gcs_path = f"gs://{BUCKET_NAME}/{AUDIO_DIR}/" + audio_file_name + ".flac"

    # 4b. Read the file from GCS
    audio_binary = tf.io.read_file(gcs_path)

    # 4c. Decode the .flac file
    # <--- CHANGE 2: USE 'tfio'
    waveform = tfio.audio.decode_flac(audio_binary, dtype=tf.int16)

    # Convert to float and normalize
    waveform = tf.cast(waveform, tf.float32) / 32768.0

    # Remove the extra channel dimension
    waveform = tf.squeeze(waveform, axis=-1)

    # 4d. Create the Spectrogram
    stft = tf.signal.stft(
        waveform,
        frame_length=N_FFT,
        frame_step=HOP_LENGTH,
        fft_length=N_FFT
    )
    spectrogram = tf.abs(stft)

    # 4e. Create the Mel Spectrogram
    mel_filterbank = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=stft.shape[-1],
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=0.0,
        upper_edge_hertz=(SAMPLE_RATE / 2.0)
    )
    mel_spectrogram = tf.tensordot(spectrogram, mel_filterbank, 1)

    # 4f. Convert to Log-Scale (dB)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    # 4g. Return the processed "image" and its label
    return log_mel_spectrogram, label

# --- 5. Build the Final Pipeline ---
BATCH_SIZE = 32

dataset = (
    dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(buffer_size=len(train_df))
    .padded_batch(BATCH_SIZE, padded_shapes=([None, N_MELS], []))
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

print("✅ tf.data pipeline built successfully.")
print(dataset)

✅ tf.data pipeline built successfully.
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


### Validation

In [2]:
# -- Validation Dataset Labels --

import pandas as pd
import os

# --- 1. Define Your Bucket and File Paths ---
BUCKET_NAME = "asvspoof-la-data-bucket"
PROTOCOL_DIR = "ASVspoof2019_LA_cm_protocols"
# This time, we use the 'dev' protocol file
DEV_PROTOCOL_FILE = "ASVspoof2019.LA.cm.dev.trl.txt"
KEY_FILE_NAME = 'temp_key.json' # We'll re-use the key

# --- Create the key file (in case it was deleted) ---
# This is a good safety check
try:
    key_string = userdata.get('GCP_KEY')
    key_data = json.loads(key_string)
    with open(KEY_FILE_NAME, 'w') as f:
        json.dump(key_data, f)
except NameError:
    print("Assuming key file already exists...")


# Construct the full GCS path
gcs_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/{DEV_PROTOCOL_FILE}"
print(f"Reading validation labels from: {gcs_path}\n")

# Explicitly pass the token to pandas
storage_options = {'token': KEY_FILE_NAME}

# --- 2. Read the Protocol File ---
# The README says the dev file has the same 5-column format
column_names = ['SPEAKER_ID', 'AUDIO_FILE_NAME', 'BLANK', 'SYSTEM_ID', 'KEY']

val_df = pd.read_csv(
    gcs_path,
    sep=' ',
    header=None,
    names=column_names,
    storage_options=storage_options
)

# --- 3. Clean Up and Process the DataFrame ---
val_df = val_df[['AUDIO_FILE_NAME', 'KEY']]
val_df['LABEL'] = val_df['KEY'].apply(
    lambda x: 0 if x == 'bonafide' else 1
)
val_df = val_df.drop(columns=['KEY'])

# --- 4. Inspect the Result ---
print("Successfully loaded and processed validation labels:")
print(val_df.info())
print("\nFirst 5 rows:")
print(val_df.head())

Reading validation labels from: gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt

Successfully loaded and processed validation labels:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24844 entries, 0 to 24843
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   AUDIO_FILE_NAME  24844 non-null  object
 1   LABEL            24844 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 388.3+ KB
None

First 5 rows:
  AUDIO_FILE_NAME  LABEL
0    LA_D_1047731      0
1    LA_D_1105538      0
2    LA_D_1125976      0
3    LA_D_1293230      0
4    LA_D_1340209      0


# CNN Model

# RNN Model